In [2]:
import hashlib
from datetime import datetime
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit

# 1. Environment Provenance (Section 2.3)
print(f"Executed on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
RANDOM_SEED = 1

# 2. Raw Ingestion & SHA-256 Hash (Section 2.1 & 4.1)
# Verifies testing against the identical dataset
raw_path = "data/raw/diamonds.csv"
hasher = hashlib.sha256()
with open(raw_path, "rb") as f:
  hasher.update(f.read())
#opens file and closes it after
print(f"Dataset SHA-256: {hasher.hexdigest()}")

raw_df = pd.read_csv(raw_path)
if "Unnamed: 0" in raw_df.columns:
  raw_df = raw_df.drop(columns=["Unnamed: 0"])

print(f"Raw shape: {raw_df.shape} | Duplicates: {raw_df.duplicated().sum()}")
#(row, columns)

# 3. Quality Flags - No silent dropping (Section 4.2 & 4.3)
audit_df = raw_df.copy()
audit_df.insert(0, "diamond_id", np.arange(1, len(audit_df) + 1))
audit_df["flag_non_positive_price"] = audit_df["price"] <= 0
audit_df["flag_zero_dim"] = (
    (audit_df["x"] <= 0) | (audit_df["y"] <= 0) | (audit_df["z"] <= 0)
)
ratio_xy = audit_df["x"] / audit_df["y"].replace(0, np.nan)
ratio_yz = audit_df["y"] / audit_df["z"].replace(0, np.nan)
audit_df["flag_extreme_aspect_ratio"] = (
    (ratio_xy < 0.9) | (ratio_xy > 1.1) |
    (ratio_yz < 1.2) | (ratio_yz > 2.1)
).fillna(False)
#diamond's are all round so x/y ratio is expected to between 0.98 to 1.02, < 0.90 or > 1.10 is extreme
#y/z ratio is expected to be between 1.50 to 1.70, < 1.20 or > 2.10 are extreme boundaries
audit_df["flag_depth_inconsistency"] = (
    audit_df["depth"] - 200 * audit_df["z"] / (audit_df["x"] + audit_df["y"])
).abs() > 5.0
#if the calculated diamond depth contridacts the measured depth by more than 5% points

flag_cols = [c for c in audit_df.columns if c.startswith("flag_")]
audit_summary = audit_df[flag_cols].sum().to_frame(name="Flagged_Count")
#sums the flag count of each column seperately and turns into dataframe with column "Flagged_Count"
total_flagged_count = audit_df[flag_cols].any(axis=1).sum()
audit_summary["Flagged_Pct"] = (audit_summary["Flagged_Count"] / len(audit_df)) * 100
audit_summary.loc["total_flagged_count"] = [total_flagged_count, (total_flagged_count / len(audit_df)) * 100]
print("\n--- Data Audit Summary ---")
print(audit_summary)

# Primary clean sample excludes physically impossible zeros and severe dimension errors (Section 12.6)
valid_mask = ~(audit_df["flag_zero_dim"] | audit_df["flag_non_positive_price"])
clean_df = audit_df[valid_mask].copy().reset_index(drop=True)

# Inspect flagged vs unflagged characteristics (Section 4.3 & 8.4)
inspect_cols = ["diamond_id", "carat", "cut", "price", "x", "y", "z", "depth"]

print("\n--- Extreme Aspect Ratio Diamonds Sample ---")
display(clean_df.loc[clean_df["flag_extreme_aspect_ratio"], inspect_cols])

print("\n--- Depth Inconsistency Diamonds Sample ---")
display(clean_df.loc[clean_df["flag_depth_inconsistency"], inspect_cols])

# Check if flagged diamonds have systematically different average prices
inspection_summary = pd.DataFrame({
    "Group": ["Normal", "Extreme Aspect Ratio", "Depth Inconsistent"],
    "Count": [
        len(clean_df[~clean_df["flag_extreme_aspect_ratio"] & ~clean_df["flag_depth_inconsistency"]]),
        clean_df["flag_extreme_aspect_ratio"].sum(),
        clean_df["flag_depth_inconsistency"].sum()
    ],
    "Mean Price ($)": [
        clean_df[~clean_df["flag_extreme_aspect_ratio"] & ~clean_df["flag_depth_inconsistency"]]["price"].mean(),
        clean_df[clean_df["flag_extreme_aspect_ratio"]]["price"].mean(),
        clean_df[clean_df["flag_depth_inconsistency"]]["price"].mean()
    ]
}).round(2)

display(inspection_summary)

Executed on: 2026-11-07 12:25:26
Dataset SHA-256: fc2f171cc18eae2138d01dcca7179db3bb30ff047dceae4467a056d52133810a
Raw shape: (53940, 10) | Duplicates: 146

--- Data Audit Summary ---
                           Flagged_Count  Flagged_Pct
flag_non_positive_price              0.0     0.000000
flag_zero_dim                       20.0     0.037078
flag_extreme_aspect_ratio           23.0     0.042640
flag_depth_inconsistency            40.0     0.074156
total_flagged_count                 53.0     0.098257

--- Extreme Aspect Ratio Diamonds Sample ---


,diamond_id,carat,cut,price,x,y,z,depth
2273,2275,1.22,Premium,3156,6.79,4.24,3.76,62.6
14021,14030,1.22,Ideal,5703,6.90,5.92,4.22,61.1
14336,14345,1.22,Ideal,5807,6.90,5.92,4.22,61.1
14627,14636,1.07,Ideal,5909,6.62,6.67,1.07,60.6
15227,15236,1.12,Premium,6115,6.64,4.11,3.70,61.6
17817,17827,1.51,Fair,7188,7.08,4.70,4.00,65.6
19493,19503,1.57,Fair,8133,7.85,5.75,3.87,67.3
20685,20695,1.53,Ideal,8971,7.43,7.50,1.53,61.9
21645,21655,1.41,Ideal,9752,7.31,7.22,1.41,60.7
24058,24068,2.00,Premium,12210,8.09,58.90,8.06,58.9



--- Depth Inconsistency Diamonds Sample ---


,diamond_id,carat,cut,price,x,y,z,depth
2273,2275,1.22,Premium,3156,6.79,4.24,3.76,62.6
2939,2942,1.00,Very Good,3293,6.26,6.19,3.19,62.8
4516,4519,1.00,Fair,3634,6.32,6.27,3.97,43.0
4685,4688,0.81,Premium,3674,5.99,5.94,3.97,61.5
6337,6342,1.00,Fair,4032,6.31,6.24,4.12,44.0
10372,10378,1.09,Ideal,4778,6.53,6.55,4.12,43.0
14627,14636,1.07,Ideal,5909,6.62,6.67,1.07,60.6
15227,15236,1.12,Premium,6115,6.64,4.11,3.70,61.6
17584,17594,1.00,Fair,7083,6.77,6.71,4.38,55.9
19493,19503,1.57,Fair,8133,7.85,5.75,3.87,67.3


,Group,Count,Mean Price ($)
0,Normal,53887,3929.93
1,Extreme Aspect Ratio,22,5032.36
2,Depth Inconsistent,27,5385.19


In [ ]:
invalid_data_mask = (
    audit_df["flag_zero_dim"] | 
    audit_df["flag_non_positive_price"] | 
    (audit_df["y"] >= 15) | 
    (audit_df["z"] >= 15) |
    ((audit_df["carat"] >= 1.0) & (audit_df["z"] < 2.0))
    # Catches the way physically impossible (way to small) z values (e.g. z < 2.0 when carat >= 1.0)
)

valid_mask = ~invalid_data_mask
clean_df = audit_df[valid_mask].copy().reset_index(drop=True)

# 4. Ordinal Score Encoding (Appendix A)
cut_map = {"Fair": 1, "Good": 2, "Very Good": 3, "Premium": 4, "Ideal": 5}
color_map = {"J": 1, "I": 2, "H": 3, "G": 4, "F": 5, "E": 6, "D": 7}
clarity_map = {"I1": 1, "SI2": 2, "SI1": 3, "VS2": 4, "VS1": 5, "VVS2": 6, "VVS1": 7, "IF": 8,}
#assumes that spacing / value between each label is linear -- will evaluate later to test the equal-spacing assumption

clean_df["cut_score"] = clean_df["cut"].map(cut_map)
clean_df["color_score"] = clean_df["color"].map(color_map)
clean_df["clarity_score"] = clean_df["clarity"].map(clarity_map)
clean_df["log_price"] = np.log(clean_df["price"])

# 5. Stratified Split to Prevent Target Leakage (Section 6.1 & 6.2)
#slicing diamonds into deciles to preserve distribution and reduce distributional shift

clean_df["price_strata"] = pd.qcut(clean_df["log_price"], q=10, labels=False)
#taking log price stabilizes heavy tail (with high prices) -- function assigns each diamond an integer from 0 to 9 (for splitting))

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_SEED)
#split into set seed so experiment is repeatable and records can be logged
train_idx, test_idx = next(splitter.split(clean_df, clean_df["price_strata"]))

dev_df = clean_df.iloc[train_idx].drop(columns=["price_strata"]).copy()
test_df = clean_df.iloc[test_idx].drop(columns=["price_strata"]).copy()
#drops column because useless now

# Verify log_price quantiles match across splits
distribution_check = pd.DataFrame({
    "Full Clean": clean_df["log_price"].describe(),
    "Dev Set (70%)": dev_df["log_price"].describe(),
    "Test Set (30%)": test_df["log_price"].describe(),
})
print(distribution_check.round(3))

counting_dups = [c for c in clean_df.columns if c != "diamond_id" and c != "price_strata"]
print(f"Duplicates Full: {clean_df.duplicated(subset=counting_dups).sum()} | "
      f"Duplicates Dev: {dev_df.duplicated(subset=counting_dups).sum()} | "
      f"Duplicates Test: {test_df.duplicated(subset=counting_dups).sum()}")

# 6. Save Separate Artifacts (Locked Test Set)
# dev_df.to_csv("data/processed/dev_data.csv", index=False)
# test_df.to_csv("data/processed/test_locked.csv", index=False)
# print(f"\nDev Set (70%): {dev_df.shape} saved to data/processed/dev_data.csv")
# print(f"Test Set (30%): {test_df.shape} locked into data/processed/test_locked.csv")
# audit_df.to_csv("data/processed/audit_trail.csv", index=False)

       Full Clean  Dev Set (70%)  Test Set (30%)
count   53914.000      37739.000       16175.000
mean        7.786          7.787           7.785
std         1.015          1.015           1.013
min         5.787          5.787           5.787
25%         6.855          6.856           6.855
50%         7.784          7.784           7.784
75%         8.580          8.581           8.577
max         9.843          9.843           9.841
Duplicates Full: 145 | Duplicates Dev: 78 | Duplicates Test: 15
